# Decomposed Pipeline — Bedrock Development and Validation Evaluation

This presentation notebook evaluates the LLM-based Research → Math → Verifier pipeline with Amazon Nova Lite. AMZN is the development batch and GOOGL is the validation batch. Gold answers are used only after inference. The four diagnostic examples are separate from the two scored 16-question batches.

## Setup

Load the reusable runtime and evaluation code from `src/`.

In [1]:
import sys
import warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="IProgress not found.*")
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

In [2]:
from agent.data import load_question
from agent.decomposed import DecomposedAgentHarness
from agent.observation import format_observable_timeline, format_run_summary
from agent.tools import ProjectTools

In [3]:
from agent.data import load_gold_questions
from pandas import DataFrame
from evaluation.metrics import (
    evaluate_saved_results,
    format_evaluation_summary,
    save_evaluation_results, save_run_records,
)

In [4]:
from config import (
    BEDROCK_NOVA_LITE_MODEL_ID, NOVA_LITE_INPUT_USD_PER_MILLION_TOKENS,
    NOVA_LITE_OUTPUT_USD_PER_MILLION_TOKENS,
)

## Development benchmark

Load the frozen AMZN questions. Gold answers remain separate from agent inputs and appear only in answer comparisons.

In [5]:
ticker = "AMZN"
gold_records = load_gold_questions(ticker)
gold_by_id = {record["question_id"]: record for record in gold_records}

In [6]:
DataFrame([
    {"question_id": record["question_id"], "category": record["category"]}
    for record in gold_records
])

,question_id,category
0,amzn-01,direct_extraction
1,amzn-02,direct_extraction
2,amzn-03,direct_extraction
3,amzn-04,direct_extraction
4,amzn-05,single_formula
5,amzn-06,single_formula
6,amzn-07,single_formula
7,amzn-08,single_formula
8,amzn-09,period_comparison
9,amzn-10,period_comparison


## Preflight

Confirm that Bedrock, Langfuse, and the local Qdrant filing index are available before spending model calls.

In [7]:
harness = DecomposedAgentHarness(
    provider="bedrock", model_id=BEDROCK_NOVA_LITE_MODEL_ID
)
preflight = harness.preflight()

In [8]:
preflight

{'ready': True,
 'provider': 'bedrock',
 'model': 'amazon.nova-lite-v1:0',
 'architecture': 'decomposed',
 'pipeline_variant': 'verification_with_repair',
 'reasoning': {'research': 'none', 'math': 'none', 'verifier': 'none'},
 'langfuse': 'authenticated',
 'qdrant_collection': 'sec_filing_chunks_v1'}

## Question 1: direct extraction

Run one question, then inspect its complete audit trail before continuing.

In [9]:
question_1 = load_question("amzn-01")
gold_1 = gold_by_id[question_1.question_id]["answer"]
run_1 = harness.run_question(question_1.question)

In [10]:
run_1.answer

MonolithicAnswer(answer_text="Amazon.com, Inc.'s consolidated net sales in fiscal 2025 was $716,924,000,000.", values=[AnswerValue(label='consolidated net sales', value='716924000000', unit='USD', period='FY2025', source_evidence_ids=['fact-amzn-revenue-fy2025'])], evidence_ids=['fact-amzn-revenue-fy2025'], calculations=[CalculationRecord(operation='add', inputs=['716924000000'], result='716924000000')], conclusions=[])

### Full trace: Question 1

In [11]:
print(format_observable_timeline(run_1.observable_timeline))

===== 01 SYSTEM PROMPT [research] attempt 1 =====

You are the research stage of a read-only SEC filing
pipeline. Use only the supplied frozen-data tools. Do not give financial advice,
use outside knowledge, calculate, or answer the user directly.

- Use evidence only for the company in the question. Match its
  requested period and unit; before calculating, confirm every input has the
  requested period. Do not guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.

Return the exact selected values, units, periods, and only evidence IDs returned by
the tools. Describe the calculation the math stage must perform, but do not perform
it yourself. If evidence is missing or ambiguous, return an empty facts list and
explain that in calculation_request.


===== 02 TOOL DEFINITIONS [research] attempt 1 =====

- search_filing: Search the frozen 10-K for

### Run summary: Question 1

In [12]:
print(format_run_summary(run_1, gold_1))

Question: What was Amazon.com, Inc.'s consolidated net sales in fiscal 2025?
Model answer: Amazon.com, Inc.'s consolidated net sales in fiscal 2025 was $716,924,000,000.
Gold answer: 716924000000
Gold values: -
Tools: lookup_fact (ok)
Evidence: fact-amzn-revenue-fy2025
Latency: 9713.91 ms | Tokens: {'inputTokens': 5407, 'outputTokens': 695, 'totalTokens': 6102} | Provider retries: 0
Structured-output repairs: 0 (no schema-driven model reinvocation)
Cost: {'input': '0.00032442', 'output': '0.0001668', 'total': '0.00049122'}
Trace: https://cloud.langfuse.com/project/cmrva8h6h0hchad0d3guy659p/traces/a880d20979dcd8987fb4837953839d3c


### Cost breakdown: Question 1

Nova Lite on-demand text pricing in `us-east-1`, frozen on 2026-08-24.

In [13]:
input_cost = run_1.token_usage["inputTokens"] * NOVA_LITE_INPUT_USD_PER_MILLION_TOKENS / 1_000_000
output_cost = run_1.token_usage["outputTokens"] * NOVA_LITE_OUTPUT_USD_PER_MILLION_TOKENS / 1_000_000
input_cost + output_cost

Decimal('0.00049122')

## Question 2: single formula

Run one formula question, then inspect it before moving on.

In [14]:
question_2 = load_question("amzn-05")
gold_2 = gold_by_id[question_2.question_id]["answer"]
run_2 = harness.run_question(question_2.question)

### Full trace: Question 2

In [15]:
print(format_observable_timeline(run_2.observable_timeline))

===== 01 SYSTEM PROMPT [research] attempt 1 =====

You are the research stage of a read-only SEC filing
pipeline. Use only the supplied frozen-data tools. Do not give financial advice,
use outside knowledge, calculate, or answer the user directly.

- Use evidence only for the company in the question. Match its
  requested period and unit; before calculating, confirm every input has the
  requested period. Do not guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.

Return the exact selected values, units, periods, and only evidence IDs returned by
the tools. Describe the calculation the math stage must perform, but do not perform
it yourself. If evidence is missing or ambiguous, return an empty facts list and
explain that in calculation_request.


===== 02 TOOL DEFINITIONS [research] attempt 1 =====

- search_filing: Search the frozen 10-K for

### Run summary: Question 2

In [16]:
print(format_run_summary(run_2, gold_2))

Question: What was Amazon.com, Inc.'s operating margin in fiscal 2025?
Model answer: Amazon.com, Inc.'s operating margin in fiscal 2025 was 11.16%.
Gold answer: 11.16
Gold values: -
Tools: lookup_fact (ok), lookup_fact (ok), calculate (ok), calculate (failed), calculate (ok)
Evidence: fact-amzn-operating-income-fy2025, fact-amzn-revenue-fy2025
Latency: 12257.37 ms | Tokens: {'inputTokens': 11180, 'outputTokens': 1584, 'totalTokens': 12764} | Provider retries: 0
Structured-output repairs: 1 (schema feedback reinvoked the model within the same agent invocation)
Cost: {'input': '0.0006708', 'output': '0.00038016', 'total': '0.00105096'}
Trace: https://cloud.langfuse.com/project/cmrva8h6h0hchad0d3guy659p/traces/d85d3e95c54c967486bc0dfd4e1e6006


## Question 3: period comparison

This example combines filing search, normalized facts, and calculations.

In [17]:
question_3 = load_question("amzn-12")
gold_3 = gold_by_id[question_3.question_id]["answer"]
run_3 = harness.run_question(question_3.question)

### Full trace: Question 3

In [18]:
print(format_observable_timeline(run_3.observable_timeline))

===== 01 SYSTEM PROMPT [research] attempt 1 =====

You are the research stage of a read-only SEC filing
pipeline. Use only the supplied frozen-data tools. Do not give financial advice,
use outside knowledge, calculate, or answer the user directly.

- Use evidence only for the company in the question. Match its
  requested period and unit; before calculating, confirm every input has the
  requested period. Do not guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.

Return the exact selected values, units, periods, and only evidence IDs returned by
the tools. Describe the calculation the math stage must perform, but do not perform
it yourself. If evidence is missing or ambiguous, return an empty facts list and
explain that in calculation_request.


===== 02 TOOL DEFINITIONS [research] attempt 1 =====

- search_filing: Search the frozen 10-K for

### Run summary: Question 3

In [19]:
print(format_run_summary(run_3, gold_3))

Question: From fiscal 2024 to fiscal 2025, how did AWS operating income growth compare with Amazon's consolidated operating income growth?
Model answer: From fiscal 2024 to fiscal 2025, AWS operating income grew by 87.34% while Amazon's consolidated operating income grew by 85.77%. AWS operating income grew at a faster rate than Amazon's consolidated operating income.
Gold answer: consolidated operating income grew faster than AWS operating income; the growth-rate difference was 2.10 percentage points
Gold values: AWS operating income growth: 14.49 percent, consolidated operating income growth: 16.59 percent
Tools: lookup_fact (ok), lookup_fact (ok), search_filing (ok), search_filing (ok), calculate (ok), calculate (ok)
Evidence: fact-amzn-operating-income-fy2024, fact-amzn-operating-income-fy2025, filing-amzn-2025-page-0027-table-01, filing-amzn-2025-page-0027-table-01
Latency: 15048.03 ms | Tokens: {'inputTokens': 19035, 'outputTokens': 1733, 'totalTokens': 20768} | Provider retries:

## Question 4: multi-step calculation

The final dry run checks a longer chain before the complete batch.

In [20]:
question_4 = load_question("amzn-16")
gold_4 = gold_by_id[question_4.question_id]["answer"]
run_4 = harness.run_question(question_4.question)

### Full trace: Question 4

In [21]:
print(format_observable_timeline(run_4.observable_timeline))

===== 01 SYSTEM PROMPT [research] attempt 1 =====

You are the research stage of a read-only SEC filing
pipeline. Use only the supplied frozen-data tools. Do not give financial advice,
use outside knowledge, calculate, or answer the user directly.

- Use evidence only for the company in the question. Match its
  requested period and unit; before calculating, confirm every input has the
  requested period. Do not guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.

Return the exact selected values, units, periods, and only evidence IDs returned by
the tools. Describe the calculation the math stage must perform, but do not perform
it yourself. If evidence is missing or ambiguous, return an empty facts list and
explain that in calculation_request.


===== 02 TOOL DEFINITIONS [research] attempt 1 =====

- search_filing: Search the frozen 10-K for

### Run summary: Question 4

In [22]:
print(format_run_summary(run_4, gold_4))

Question: Did Amazon's net sales, operating income, operating cash flow, and total assets grow at similar rates from fiscal 2024 to fiscal 2025?
Model answer: Amazon's net sales, operating income, operating cash flow, and total assets did not grow at similar rates from fiscal 2024 to fiscal 2025. While net sales, operating income, and operating cash flow all showed growth, the growth rate of total assets was significantly higher.
Gold answer: all four measures increased; total assets grew fastest; net sales grew slowest; the spread between the fastest and slowest growth rates was 18.53 percentage points
Gold values: net sales growth: 12.38 percent, operating income growth: 16.59 percent, operating cash flow growth: 20.40 percent, total assets growth: 30.91 percent
Tools: lookup_fact (ok), lookup_fact (ok), lookup_fact (ok), lookup_fact (ok), calculate (ok), calculate (ok), calculate (ok), calculate (ok), calculate (ok), calculate (ok), calculate (ok), calculate (ok), calculate (ok), ca

## Full AMZN development batch

Run all 16 development questions with Nova Lite under the same prompt, tools, retrieval, and model settings used for validation. These batch records—not the diagnostic examples above—are saved and evaluated together.

In [23]:
batch_runs = harness.run_batch(ticker)
assert [run.question.question_id for run in batch_runs] == [gold["question_id"] for gold in gold_records]
results_dir = save_run_records(batch_runs)

In [24]:
DataFrame([
    {"question_id": run.question.question_id, "status": run.status}
    for run in batch_runs
])

,question_id,status
0,amzn-01,success
1,amzn-02,success
2,amzn-03,failure
3,amzn-04,success
4,amzn-05,success
5,amzn-06,success
6,amzn-07,failure
7,amzn-08,failure
8,amzn-09,success
9,amzn-10,success


## Inspect every answer

The compact view makes incorrect values, missing evidence, failed calls, and unexpectedly expensive questions visible before looking at aggregate scores.

In [25]:
for run in batch_runs:
    gold = gold_by_id[run.question.question_id]["answer"]
    print(format_run_summary(run, gold))
    print("-" * 88)

Question: What was Amazon.com, Inc.'s consolidated net sales in fiscal 2025?
Model answer: Amazon.com, Inc.'s consolidated net sales in fiscal 2025 were $716,924,000,000.
Gold answer: 716924000000
Gold values: -
Tools: lookup_fact (ok)
Evidence: fact-amzn-revenue-fy2025
Latency: 6554.18 ms | Tokens: {'inputTokens': 7404, 'outputTokens': 791, 'totalTokens': 8195} | Provider retries: 0
Structured-output repairs: 1 (schema feedback reinvoked the model within the same agent invocation)
Cost: {'input': '0.00044424', 'output': '0.00018984', 'total': '0.00063408'}
Trace: https://cloud.langfuse.com/project/cmrva8h6h0hchad0d3guy659p/traces/85931b387d880b3e05764317fcb8f69a
----------------------------------------------------------------------------------------
Question: What was Amazon.com, Inc.'s consolidated operating income in fiscal 2025?
Model answer: Amazon.com, Inc.'s consolidated operating income in fiscal 2025 was $79,975,000,000.
Gold answer: 79975000000
Gold values: -
Tools: lookup_fa

## Deterministic evaluation

Gold values, calculation steps, required tools, and saved tool events are compared with ordinary Python. Structured final values must match the gold value at its reported precision and use the same unit; extra final values fail. Formatting is scored separately across structured values and answer text.

Retrieval is measured from saved `search_filing` events. Each search call contributes a top-five row; aggregate recall gives a question credit when any of its searches finds relevant evidence, while irrelevant retrieval is averaged by question. Haiku 4.5 adds exploratory 1–5 workflow scores without overriding deterministic numerical correctness.

In [26]:
saved_evaluation = evaluate_saved_results(
    latest=True, architecture=batch_runs[0].architecture, ticker=ticker,
)
assert saved_evaluation["results_directory"] == results_dir
question_metrics = saved_evaluation["question_rows"]
retrieval_metrics = saved_evaluation["retrieval_rows"]
summary = saved_evaluation["summary"]

## Exploratory Haiku 4.5 workflow review

These qualitative scores support diagnosis only. They cannot override deterministic numerical or audit failures.

In [27]:
saved_evaluation = evaluate_saved_results(
    latest=True, rerun_judge=True,
    architecture=batch_runs[0].architecture, ticker=ticker,
)
assert saved_evaluation["results_directory"] == results_dir
judge_metrics = saved_evaluation["judge_rows"] or []
summary = saved_evaluation["summary"]

In [28]:
DataFrame(judge_metrics, columns=(
    "question_id", "llm_answer_correctness_grade", "workflow_correctness",
    "chronological_coherence", "workflow_efficiency", "auditability",
))

,question_id,llm_answer_correctness_grade,workflow_correctness,chronological_coherence,workflow_efficiency,auditability
0,amzn-01,5,5,5,5,5
1,amzn-02,5,5,5,5,5
2,amzn-03,1,1,1,1,2
3,amzn-04,2,2,4,4,3
4,amzn-05,5,5,5,3,5
5,amzn-06,5,4,5,5,4
6,amzn-07,1,2,4,1,4
7,amzn-08,2,2,3,3,3
8,amzn-09,3,4,5,4,4
9,amzn-10,3,3,5,4,3


### AMZN questions needing review

Only runs with a failed answer, evidence, period, formatting, or audit check appear here. Formula failures also make the audit check fail.

In [29]:
CHECKS = ("numerical_correct", "evidence_valid", "evidence_complete",
          "value_evidence_valid", "periods_valid", "formatting_correct",
          "audit_trail_complete")
failed_question_metrics = [row for row in question_metrics if row["status"] != "success"
                           or any(row[name] is not True for name in CHECKS)]
DataFrame(failed_question_metrics, columns=(
    "question_id", "category", "numerical_correct",
    "evidence_complete", "formula_correct", "formatting_correct",
    "audit_trail_complete",
))

,question_id,category,numerical_correct,evidence_complete,formula_correct,formatting_correct,audit_trail_complete
0,amzn-03,direct_extraction,False,False,None,False,False
1,amzn-04,direct_extraction,False,True,None,False,False
2,amzn-06,single_formula,True,True,False,True,False
3,amzn-07,single_formula,False,False,False,False,False
4,amzn-08,single_formula,False,False,False,False,False
5,amzn-09,period_comparison,True,True,False,True,False
6,amzn-10,period_comparison,True,True,False,True,False
7,amzn-11,period_comparison,False,True,False,True,False
8,amzn-12,period_comparison,False,True,False,True,False
9,amzn-13,multi_step,False,True,False,True,False


### Metrics by question type

Category scores show whether errors are concentrated in extraction, one formula, comparisons, or longer calculation chains.

In [30]:
DataFrame(summary["by_category"]).T.rename_axis("question_type")

,run_count,numerical_correctness_rate,evidence_completeness_rate,formula_correctness_rate,formatting_correctness_rate
question_type,,,,,
direct_extraction,4.0,0.5,0.75,NaN,0.5
multi_step,4.0,0.0,1.00,0.00,0.5
period_comparison,4.0,0.5,1.00,0.00,1.0
single_formula,4.0,0.5,0.50,0.25,0.5


### Final metrics

Higher is better except for unsupported-number rate and irrelevant retrieval rate. Evidence validity checks whether cited answer IDs were returned by tools. Evidence completeness checks whether every final value lists source IDs included in the answer citations. Retrieval rows score each search call's top five, then aggregate results by question.

In [31]:
print(format_evaluation_summary(summary))

Runs: 16
run_success_rate: 81.25%
numerical_correctness_rate: 37.50%
evidence_validity_rate: 81.25%
evidence_completeness_rate: 81.25%
formula_correctness_rate: 8.33%
formatting_correctness_rate: 62.50%
unsupported_number_rate: 42.86%
audit_trail_completeness_rate: 18.75%
math_repair_invocation_rate: 0.00%
math_repair_success_rate: 0.00%
tool_precision: 87.88%
tool_recall: 96.67%
tool_f1: 92.06%
tool_exact_match_rate: 75.00%
tool_invocation_rate: 100.00%
tool_execution_success_rate: 96.97%
conditional_execution_rate: 89.58%
evidence_recall_at_5: 75.00%
irrelevant_retrieval_rate_at_5: 70.00%
judge_success_rate: 100.00%
judge_cost_coverage_rate: 100.00%
structured_output_repair_rate: 31.25%
Average latency: 12231.22 ms
Input tokens: 209344
Output tokens: 25681
Reasoning tokens: 0
Retries: 0
Structured-output repairs: 7
Infrastructure failures: 0
Estimated cost: $0.01872408
Judge llm answer correctness grade: 2.56/5
Judge workflow correctness: 2.69/5
Judge chronological coherence: 3.88/5


### AMZN findings and conclusion

In this recorded decomposed Nova Lite batch, 13 of 16 runs produced an approved answer. Numerical correctness was 37.50%, Tool F1 was 92.06%, Evidence Recall@5 was 75.00%, and the irrelevant-retrieval rate was 70.00%. Direct extraction, single-formula, and period-comparison questions each achieved 2/4 numerical correctness; none of the four multi-step questions passed.

Formula correctness was 8.33%, audit-trail completeness was 18.75%, and the verifier requested no Math repair. The review table contains thirteen questions, including numerically correct answers whose recorded calculation path was incomplete or non-canonical.

Every AMZN review row explained:

- **AMZN-03 — fails run status, numerical, evidence completeness, formatting, and audit; formula is not applicable:** Research returned consolidated operating-income facts but no segment filing table. The verifier rejected the question because it could not determine the largest segment.
- **AMZN-04 — fails numerical, formatting, and audit:** the correct filing evidence was available, but the answer retained the table scale as `4750 million USD` instead of `4750000000 USD`.
- **AMZN-06 — numerical and formatting pass; formula and audit fail:** the final 10.83% net margin is correct, but Math divided first and then called `percentage(ratio, 100)`, which returned `0.1083...`, not 10.83. The final value was not reproduced by a valid calculator event.
- **AMZN-07 — fails run status and every scored answer/audit check:** Math correctly divided cash flow by revenue, then used the percentage tool incorrectly and repeatedly called `add(value, 0)` until the turn limit. No structured answer was produced.
- **AMZN-08 — fails run status and every scored answer/audit check:** the calculator obtained the correct 179.624...%, but the verifier could not populate `source_evidence_ids`; after repairs, the run was rejected with no final answer.
- **AMZN-09 — numerical and formatting pass; formula and audit fail:** the final 12.38% and 16.59% growth rates are correct, but the trace omitted the expected final 4.22-percentage-point comparison and included failed formula-string calls.
- **AMZN-10 — numerical and formatting pass; formula and audit fail:** 31.09% and 20.40% are correct, but the expected 10.69-percentage-point difference was never calculated; unnecessary new/old divisions were also recorded.
- **AMZN-11 — fails numerical, formula, and audit:** the dollar increase of `$193,148,000,000` is correct, but Math and the verifier treated the new/old ratio 130.91% as percentage growth. Gold requires 30.91%.
- **AMZN-12 — fails numerical, formula, and audit:** Math calculated new/old ratios of 114.49% and 116.59%, then the answer incorrectly converted them to 11.49% and 11.66%. Gold growth is 14.49% and 16.59%.
- **AMZN-13 — fails numerical, formula, and audit:** the answer returned relative changes of 3.75% and 16.65% rather than the four required annual margins: 10.75%, 11.16%, 9.29%, and 10.83%.
- **AMZN-14 — fails numerical, formula, and audit:** it returned only a 1.05% decrease. Gold requires both annual AWS shares, 58.07% and 57.03%, with the change interpreted as percentage points.
- **AMZN-15 — fails numerical, formula, formatting, and audit:** the `$26,068,333,333.33` difference is present, but the answer omits the `$113,445,666,666.67` average and 22.98% comparison. The text also uses the prohibited `billion` abbreviation, and the exact average was not reproduced as one calculator result.
- **AMZN-16 — fails numerical, formula, formatting, and audit:** Math used invented rounded FY2024 inputs—such as `$680B` revenue instead of `$637.959B`—and returned 5.43%, 8.07%, 16.26%, and 4.88% instead of 12.38%, 16.59%, 20.40%, and 30.91%. The prose omitted the values.

There were no AWS infrastructure failures or provider retries. The judge rated answer correctness at 2.56/5 and auditability at 3.69/5. Judge cost was $0.2256 compared with $0.0187 for the Nova pipeline. The strong tool score alongside weak numerical and audit scores indicates that evidence access was better than calculation and verification reliability.

## Save evaluation files

The run records were saved before evaluation. The following cell writes deterministic metrics, retrieval metrics, judge records, and the aggregate summary to the same result folder.

In [32]:
results_dir = save_evaluation_results(
    results_dir, question_metrics, summary, retrieval_metrics,
    judge_rows=judge_metrics or None,
)
results_dir

PosixPath('/home/atanas-vitanov/Documents/final_project_financial_agent/results/2026-08-31T221127530825Z-decomposed-verification_with_repair-amazon-nova-lite-v1-0-AMZN')

In [33]:
DataFrame({"saved_file": sorted(path.name for path in results_dir.iterdir())})

,saved_file
0,judge_records.jsonl
1,question_metrics.csv
2,retrieval_metrics.csv
3,runs.jsonl
4,summary.json


## Failures

Failed runs remain experimental results and are displayed rather than selectively rerun. Failed tool calls inside otherwise completed runs remain visible in the per-question summaries, traces, and tool-execution metric.

In [34]:
DataFrame(
    [
        {"question_id": run.question.question_id,
         "failure_category": run.failure_category,
         "failure_message": run.failure_message}
        for run in batch_runs if run.status == "failure"
    ],
    columns=("question_id", "failure_category", "failure_message"),
)

,question_id,failure_category,failure_message
0,amzn-03,verifier_rejected,The research package does not contain the nece...
1,amzn-07,math_agent_error,math stopped at limit_turns
2,amzn-08,verifier_rejected,Missing 'source_evidence_ids' field in the 'va...


# Validation batch: Alphabet (GOOGL)

Run the frozen validation company with the same model, prompts, tools, retrieval, limits, and scoring rules used for AMZN. No validation result is used to modify the pipeline inside this notebook.

In [35]:
validation_ticker = "GOOGL"
validation_gold_records = load_gold_questions(validation_ticker)
validation_gold_by_id = {
    record["question_id"]: record for record in validation_gold_records
}

In [36]:
validation_runs = harness.run_batch(validation_ticker)
assert len(validation_runs) == len(validation_gold_records)
validation_results_dir = save_run_records(validation_runs)

In [37]:
DataFrame([
    {"question_id": run.question.question_id, "status": run.status}
    for run in validation_runs
])

,question_id,status
0,googl-01,success
1,googl-02,success
2,googl-03,success
3,googl-04,success
4,googl-05,success
5,googl-06,success
6,googl-07,failure
7,googl-08,success
8,googl-09,success
9,googl-10,success


## Inspect every answer

Use the same compact answer and gold comparison as the AMZN batch.

In [38]:
for run in validation_runs:
    gold = validation_gold_by_id[run.question.question_id]["answer"]
    print(format_run_summary(run, gold))
    print("-" * 88)

Question: What were Alphabet Inc.'s consolidated revenues in fiscal 2025?
Model answer: Alphabet Inc.'s consolidated revenues in fiscal 2025 were $402,836,000,000.
Gold answer: 402836000000
Gold values: -
Tools: lookup_fact (ok)
Evidence: fact-googl-revenue-fy2025
Latency: 5809.02 ms | Tokens: {'inputTokens': 5419, 'outputTokens': 695, 'totalTokens': 6114} | Provider retries: 0
Structured-output repairs: 0 (no schema-driven model reinvocation)
Cost: {'input': '0.00032514', 'output': '0.0001668', 'total': '0.00049194'}
Trace: https://cloud.langfuse.com/project/cmrva8h6h0hchad0d3guy659p/traces/c92a94c1943d8f8db7c3b5c9e5c2c5e0
----------------------------------------------------------------------------------------
Question: What was Alphabet Inc.'s consolidated income from operations in fiscal 2025?
Model answer: Alphabet Inc.'s consolidated income from operations in fiscal 2025 was $129,039 million.
Gold answer: 129039000000
Gold values: -
Tools: search_filing (ok)
Evidence: filing-googl

## Deterministic evaluation

In [39]:
validation_saved_evaluation = evaluate_saved_results(
    latest=True, architecture=validation_runs[0].architecture,
    ticker=validation_ticker,
)
assert validation_saved_evaluation["results_directory"] == validation_results_dir
validation_question_metrics = validation_saved_evaluation["question_rows"]
validation_retrieval_metrics = validation_saved_evaluation["retrieval_rows"]
validation_summary = validation_saved_evaluation["summary"]

## Exploratory Haiku 4.5 workflow review

These qualitative scores support diagnosis only. They cannot override deterministic numerical or audit failures.

In [40]:
validation_saved_evaluation = evaluate_saved_results(
    latest=True, rerun_judge=True,
    architecture=validation_runs[0].architecture, ticker=validation_ticker,
)
assert validation_saved_evaluation["results_directory"] == validation_results_dir
validation_judge_metrics = validation_saved_evaluation["judge_rows"] or []
validation_summary = validation_saved_evaluation["summary"]

In [41]:
DataFrame(validation_judge_metrics, columns=(
    "question_id", "llm_answer_correctness_grade", "workflow_correctness",
    "chronological_coherence", "workflow_efficiency", "auditability",
))

,question_id,llm_answer_correctness_grade,workflow_correctness,chronological_coherence,workflow_efficiency,auditability
0,googl-01,5,5,5,4,5
1,googl-02,2,2,4,3,2
2,googl-03,5,5,5,4,5
3,googl-04,2,3,5,5,5
4,googl-05,5,4,5,4,4
5,googl-06,5,5,5,5,4
6,googl-07,1,2,3,2,2
7,googl-08,5,4,5,5,4
8,googl-09,3,3,5,4,4
9,googl-10,3,3,3,4,3


### GOOGL questions needing review

Only GOOGL runs with a failed answer, evidence, period, formatting, or audit check appear in this table. Formula failures also make the audit check fail.

In [42]:
validation_failed_question_metrics = [
    row for row in validation_question_metrics if row["status"] != "success"
    or any(row[name] is not True for name in CHECKS)
]
DataFrame(validation_failed_question_metrics, columns=(
    "question_id", "category", "numerical_correct",
    "evidence_complete", "formula_correct", "formatting_correct",
    "audit_trail_complete",
))

,question_id,category,numerical_correct,evidence_complete,formula_correct,formatting_correct,audit_trail_complete
0,googl-02,direct_extraction,False,True,None,False,False
1,googl-04,direct_extraction,False,True,None,False,False
2,googl-05,single_formula,True,True,False,True,False
3,googl-06,single_formula,True,True,False,True,False
4,googl-07,single_formula,False,False,False,False,False
5,googl-08,single_formula,True,True,False,True,False
6,googl-09,period_comparison,True,True,False,True,False
7,googl-10,period_comparison,True,True,False,True,False
8,googl-11,period_comparison,False,True,False,True,False
9,googl-12,period_comparison,True,True,False,True,False


### Metrics by question type

In [43]:
DataFrame(validation_summary["by_category"]).T.rename_axis("question_type")

,run_count,numerical_correctness_rate,evidence_completeness_rate,formula_correctness_rate,formatting_correctness_rate
question_type,,,,,
direct_extraction,4.0,0.50,1.00,NaN,0.50
multi_step,4.0,0.00,0.75,0.25,0.50
period_comparison,4.0,0.75,1.00,0.00,1.00
single_formula,4.0,0.75,0.75,0.00,0.75


### Final metrics

Use the same metric definitions as the AMZN section so the two company batches remain directly comparable.

In [44]:
print(format_evaluation_summary(validation_summary))

Runs: 16
run_success_rate: 87.50%
numerical_correctness_rate: 50.00%
evidence_validity_rate: 87.50%
evidence_completeness_rate: 87.50%
formula_correctness_rate: 8.33%
formatting_correctness_rate: 68.75%
unsupported_number_rate: 15.00%
audit_trail_completeness_rate: 12.50%
math_repair_invocation_rate: 0.00%
math_repair_success_rate: 0.00%
tool_precision: 84.38%
tool_recall: 90.00%
tool_f1: 87.10%
tool_exact_match_rate: 62.50%
tool_invocation_rate: 100.00%
tool_execution_success_rate: 100.00%
conditional_execution_rate: 83.33%
evidence_recall_at_5: 75.00%
irrelevant_retrieval_rate_at_5: 45.00%
judge_success_rate: 100.00%
judge_cost_coverage_rate: 100.00%
structured_output_repair_rate: 50.00%
Average latency: 11328.57 ms
Input tokens: 192603
Output tokens: 23977
Reasoning tokens: 0
Retries: 0
Structured-output repairs: 9
Infrastructure failures: 0
Estimated cost: $0.01731066
Judge llm answer correctness grade: 2.88/5
Judge workflow correctness: 2.94/5
Judge chronological coherence: 4.06/5

### GOOGL findings and conclusion

In this recorded validation batch, 14 of 16 runs produced an approved answer. Numerical correctness was 50.00%, Tool F1 was 87.10%, Evidence Recall@5 was 75.00%, and the irrelevant-retrieval rate was 45.00%. Direct extraction achieved 2/4 numerical correctness, single-formula and period-comparison questions each achieved 3/4, and no multi-step question passed.

Formula correctness was 8.33%, audit-trail completeness was 12.50%, and the verifier again requested no Math repair. The review table contains fourteen questions. Several have correct final values but remain in the table because their tool sequence did not reproduce the frozen gold calculation path.

Every GOOGL review row explained:

- **GOOGL-02 — fails numerical, formatting, and audit:** the filing value was stored as `129039 USD` and displayed as `$129,039 million`; gold requires `129039000000 USD`.
- **GOOGL-04 — fails numerical, formatting, and audit:** `13910` with the invalid structured unit `millions` was returned instead of `13910000000 USD`.
- **GOOGL-05 — numerical and formatting pass; formula and audit fail:** the final 32.03% operating margin is correct, but Math used `divide` followed by `percentage(ratio, 1)` rather than the frozen direct `percentage(operating income, revenue)` calculation.
- **GOOGL-06 — numerical and formatting pass; formula and audit fail:** the 32.81% net margin is correct, but the same alternative two-step calculation does not match the required canonical calculator event.
- **GOOGL-07 — fails run status and every scored answer/audit check:** Retrieval and Math produced the correct 40.888...%, but the verifier omitted `source_evidence_ids` and rejected the run after repairs.
- **GOOGL-08 — numerical and formatting pass; formula and audit fail:** 124.62% is correct, but it was produced through `divide` and then `percentage(ratio, 1)` rather than the expected direct percentage calculation.
- **GOOGL-09 — numerical and formatting pass; formula and audit fail:** the final 15.09% and 14.81% growth rates are correct, but the expected 0.28-percentage-point comparison was not calculated and extra new/old divisions were recorded.
- **GOOGL-10 — numerical and formatting pass; formula and audit fail:** 32.01% and 31.46% are correct, but the final 0.56-percentage-point difference was never calculated; extra ratio steps remain in the trace.
- **GOOGL-11 — fails numerical, formula, and audit:** the `$145,025,000,000` absolute change is correct, but the verifier approved the 132.21% new/old ratio instead of 32.21% growth.
- **GOOGL-12 — numerical and formatting pass; formula and audit fail:** the final 14.96% and 14.81% are correct, but segment calculator inputs remained in millions rather than absolute USD and the expected 0.15-percentage-point difference was omitted.
- **GOOGL-13 — formula passes; numerical, formatting, and audit fail:** the tool trace correctly calculated all four margins and their changes. The final answer then reused 32.81% as the 2025 operating margin, claimed a 0.70-point increase instead of a 0.08-point decline, returned only changes instead of four required margins, and used `%` rather than the structured `percent` unit.
- **GOOGL-14 — fails numerical, formula, and audit:** the pipeline answered consolidated operating-income growth—`$16,649,000,000` and 14.81%—instead of Google Services' share of consolidated operating income. Gold requires 107.89% and 108.03%.
- **GOOGL-15 — fails numerical, formula, and audit:** the correct cash-flow sum was `391758000000`, but Math later divided `351758000000` by three. It returned only `$47,460,333,333.33`; gold requires a `$130,586,000,000.00` average, `$34,127,000,000.00` difference, and 26.13%.
- **GOOGL-16 — fails run status and every scored answer/audit check:** lookup events returned the FY2024 facts, but the Research handoff omitted them. The verifier rejected the run as missing all four FY2024 inputs, so no final answer was produced.

There were no AWS infrastructure failures or provider retries. The judge rated answer correctness at 2.88/5 and auditability at 3.50/5. Judge cost was $0.2189 compared with $0.0173 for the Nova pipeline. The decomposed structure remains inspectable, but this run still does not demonstrate that the verifier reliably detects and repairs unsupported calculations.

## Save evaluation files

Write the validation metrics, retrieval metrics, judge records, and aggregate summary beside the saved GOOGL run records.

In [45]:
validation_results_dir = save_evaluation_results(
    validation_results_dir, validation_question_metrics, validation_summary,
    validation_retrieval_metrics, judge_rows=validation_judge_metrics,
)
validation_results_dir

PosixPath('/home/atanas-vitanov/Documents/final_project_financial_agent/results/2026-08-31T221833335685Z-decomposed-verification_with_repair-amazon-nova-lite-v1-0-GOOGL')

In [46]:
DataFrame({"saved_file": sorted(path.name for path in validation_results_dir.iterdir())})

,saved_file
0,judge_records.jsonl
1,question_metrics.csv
2,retrieval_metrics.csv
3,runs.jsonl
4,summary.json


## Failures

Display failed validation runs without selectively rerunning them. Failed tool calls inside completed runs remain available in their summaries and traces.

In [47]:
DataFrame(
    [
        {"question_id": run.question.question_id,
         "failure_category": run.failure_category,
         "failure_message": run.failure_message}
        for run in validation_runs if run.status == "failure"
    ],
    columns=("question_id", "failure_category", "failure_message"),
)

,question_id,failure_category,failure_message
0,googl-07,verifier_rejected,Missing 'source_evidence_ids' field for the va...
1,googl-16,verifier_rejected,"Missing fiscal 2024 values for revenues, opera..."


## Overall Nova Lite conclusion

Across AMZN and GOOGL, decomposition made the stages easier to inspect but did not remove Nova Lite's main weaknesses. The pipeline usually retrieved real evidence, while most accuracy loss occurred in Math or during the Research-to-Math-to-Verifier handoff.

The recurring model errors were unscaled filing values, confusion between ratios and percentage growth, missing comparison steps, invented or changed intermediate inputs, and incomplete multi-value answers. The decomposed design added two further failure modes: Research sometimes omitted facts that its tools had already returned, and the verifier sometimes rejected a correct calculation because required structured evidence fields were missing. More importantly, it also approved incorrect ratios or wrote a final answer that contradicted correct calculator events. No Math repair was requested in either company batch.

Numerical correctness remained strongest on direct extraction and single-formula questions and weakest on multi-step questions, where neither company recorded a pass. Some low formula scores reflect a different but mathematically valid calculator path or a missing comparison step even when the displayed numbers were correct; the per-question explanations identify those cases separately. Overall, Nova Lite's primary problem was reliable financial calculation and final-answer verification, not fabrication of evidence. The LLM verifier in this experiment was not dependable enough to guarantee finance-grade precision.